# PD vs HC — Mamba training (Deliverable 3)
Pulls the latest code from GitHub and trains on the Kaggle T4 GPU.
Data comes from the attached private dataset `pd-gait-vgrf-windows`.

**Fast iteration:** after editing code locally and `git push`, just re-run the *Pull* cell and then the *Train* cell — no reinstall, no restart.

In [ ]:
# 1) GPU check + pull latest code
import os, subprocess, torch
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2'
cap = torch.cuda.get_device_capability()
assert cap >= (7, 5), f'Need T4 (sm_75+), got sm_{cap[0]}{cap[1]} — choose GPU T4 x2, NOT P100'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__, '| cuda', torch.version.cuda)

REPO = 'https://github.com/Ahmadrezanourozii/Project-Time-Series.git'
if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, '/kaggle/working/repo'], check=True)
print(subprocess.run(['git', '-C', '/kaggle/working/repo', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

In [ ]:
# 2) Install mamba-ssm (idempotent — skipped when already importable)
import importlib.util, subprocess, sys

def try_install(*specs):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', *specs]
    print('$', ' '.join(cmd))
    return subprocess.run(cmd).returncode == 0

if importlib.util.find_spec('mamba_ssm') is None:
    ok = try_install('causal-conv1d>=1.4.0', 'mamba-ssm==2.2.2')
    if not ok or importlib.util.find_spec('mamba_ssm') is None:
        print('pinned 2.2.2 failed — trying latest release')
        ok = try_install('causal-conv1d', 'mamba-ssm')
    assert ok and importlib.util.find_spec('mamba_ssm') is not None, 'mamba-ssm install failed — see log'
import mamba_ssm
print('mamba_ssm', mamba_ssm.__version__, 'OK')

In [ ]:
# 3) SMOKE GATE — run this once per fresh session before any full run (~2-3 min)
!cd /kaggle/working/repo && CUDA_VISIBLE_DEVICES=0 python run_mamba.py --smoke --require-cuda-kernels \
    --npz /kaggle/input/pd-gait-vgrf-windows/raw_windows_v2.npz \
    --fold-json /kaggle/input/pd-gait-vgrf-windows/fold_assignments.json \
    --output-dir /kaggle/working/smoke

In [ ]:
# 4) TRAIN — edit args per experiment (E0: base model, all feet, 5 folds)
!cd /kaggle/working/repo && CUDA_VISIBLE_DEVICES=0 python run_mamba.py --require-cuda-kernels \
    --npz /kaggle/input/pd-gait-vgrf-windows/raw_windows_v2.npz \
    --fold-json /kaggle/input/pd-gait-vgrf-windows/fold_assignments.json \
    --output-dir /kaggle/working/outputs \
    --checkpoint-dir /kaggle/working/checkpoints \
    --foot all --model-size base --variant mamba2

In [ ]:
# 5) Package results for download
import json, shutil, pathlib
res = pathlib.Path('/kaggle/working/outputs/results.json')
if res.exists():
    print(json.dumps(json.loads(res.read_text())['results'], indent=1)[:3000])
    shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/outputs')
    print('\n-> /kaggle/working/results.zip')